# Neural Network with TensorFlow

## Binary Classification for Breast Cancer Diagnosis

Dataset: Breast Cancer Wisconsin (Diagnostic)
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
import sklearn
import scikeras
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [2]:
pd.__version__, tf.__version__, sklearn.__version__, scikeras.__version__

('3.0.6', '2.18.1', '1.8.0', '0.13.0')

### Load the Data

**Load the dataset and create the training and test sets with `train_test_split`, using 25% of the observations for testing.**

In [3]:
X = pd.read_csv('data/breast_features.csv')
y = pd.read_csv('data/breast_labels.csv')

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [4]:
X_treinamento.shape, y_treinamento.shape

((426, 30), (426, 1))

In [5]:
X_teste.shape, y_teste.shape

((143, 30), (143, 1))

### Neural Network Structure and Initial Test

In [6]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
import tensorflow as tf
from tensorflow.keras import backend as k

**Create the neural network with the following configuration:**

**An input layer with 30 features.**

**A dense hidden layer with 16 neurons. Why was this size selected?**

Sixteen neurons provide an intermediate size between the 30 input features and the single output. This is not a fixed rule and should be compared with other configurations.

**Use ReLU activation in the hidden layer and initialize its weights with the random uniform initializer.**

**Add an output layer with sigmoid activation. Why is sigmoid suitable for this output?**

Sigmoid constrains the output to the interval from 0 to 1. For this binary problem, the value is interpreted as the probability of the positive class.

**Why is the `Sequential` class appropriate for this network?**

The `Sequential` class is appropriate because the layers form a simple linear stack without branches.

In [7]:
k.clear_session()
rede_neural = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

**Explain the values displayed by `rede_neural.summary()`.**

- **Layer (type)**: name and type of each network layer.
- **Output Shape**: dimensions produced by each layer. `(None, 16)` means that every sample produces 16 values, while `None` allows any batch size.
- **Param #**: number of trainable parameters. The 16-neuron dense layer has (30 inputs × 16 neurons) + 16 biases = 496 parameters. The output layer has (16 × 1) + 1 bias = 17 parameters, for a total of 513.
- **Trainable params**: parameters updated during training.
- **Non-trainable params**: fixed parameters, such as parameters in frozen layers.

In [8]:
rede_neural.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 513 (2.00 KB)

 Trainable params: 513 (2.00 KB)

 Non-trainable params: 0 (0.00 B)

**Add the Adam optimizer, binary cross-entropy loss, and binary accuracy metric.**

**What do optimizers do, and how does Adam work?**

Optimizers update network weights to minimize the loss function. Adam (Adaptive Moment Estimation) combines two ideas:
- **Momentum**: accumulates past gradients to smooth updates and reduce oscillation.
- **RMSProp**: adapts the learning rate for each parameter based on the magnitude of recent gradients.
Together, these mechanisms adapt the update applied to each parameter during training.

**Train the configured network with `.fit`. With `batch_size=10`, how many batches are processed in total?**

The training set contains 426 samples (75% of 569). With `batch_size=10`, the model processes ⌈426 / 10⌉ = 43 batches per epoch × 100 epochs = 4,300 weight updates.

**Train the model for 100 epochs.**

In [9]:
rede_neural.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

In [10]:
rede_neural.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100, verbose=0)

**Generate predictions for the test data. Why does the network return values between 0 and 1?**

The sigmoid activation constrains each output to the interval from 0 to 1, which is interpreted as the probability of the positive class.

In [11]:
previsoes = rede_neural.predict(X_teste, verbose=0)

**Convert the probabilities into binary predictions with a threshold of 0.5.**

In [12]:
previsoes = previsoes > 0.5

In [13]:
previsoes

array([[False],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [False],
       [

**Interpret the network result.**

With a threshold of 0.5, each probability is converted into one of the two classes. The `evaluate` method calculates loss and accuracy on the test set.

In [14]:
rede_neural.evaluate(X_teste, y_teste, verbose=0)

[0.5020276308059692, 0.8881118893623352]

## Network Layers and Optimization

**Add a second 16-neuron dense hidden layer with ReLU activation and random uniform weight initialization. How many parameters does the network have now?**

With the second hidden layer:
- Layer 1: (30 × 16) + 16 = 496 parameters
- Layer 2: (16 × 16) + 16 = 272 parameters
- Output: (16 × 1) + 1 = 17 parameters
- **Total: 785 parameters**

In [15]:
k.clear_session()
rede_neural = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

In [16]:
rede_neural.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 785 (3.07 KB)

 Trainable params: 785 (3.07 KB)

 Non-trainable params: 0 (0.00 B)

**The optimizer now includes `learning_rate` and `clipvalue`. What do these parameters do?**

- **learning_rate (0.001)**: controls the gradient descent step size. Values that are too high can cause divergence, while very low values slow training. Adam uses 0.001 by default.
- **clipvalue (0.5)**: limits gradient values during backpropagation and helps prevent exploding gradients from destabilizing training.

In [17]:
otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])

**Evaluate the network again. Did the additional layer improve or reduce performance?**

The second layer increases model capacity but does not guarantee better test performance. If training performance improves while test performance declines, the network is likely overfitting.

In [18]:
rede_neural.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100, verbose=0)

In [19]:
previsoes = rede_neural.predict(X_teste, verbose=0)

In [20]:
rede_neural.evaluate(X_teste, y_teste, verbose=0)

[0.9273094534873962, 0.8531468510627747]

## K-Fold Cross-Validation

**Explain how the network is configured for cross-validation.**

During cross-validation, each fold is used once for validation while the remaining folds are used for training. With 10 folds, the network is trained 10 times. `KerasClassifier` exposes the model through the scikit-learn API.

**Calculate the standard deviation of the scores. What does it show?**

The standard deviation shows how much accuracy varies across folds. Higher values indicate greater sensitivity to the data split.

In [21]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [22]:
def criar_rede():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
    return rede_neural

In [23]:
rede_neural = KerasClassifier(model=criar_rede, epochs=100, batch_size=10, verbose=0)

In [24]:
resultados = cross_val_score(estimator=rede_neural, X=X, y=y, cv=10, scoring='accuracy')

In [25]:
resultados

array([0.75438596, 0.78947368, 0.9122807 , 0.73684211, 0.87719298,
       0.8245614 , 0.80701754, 0.85964912, 0.89473684, 0.78571429])

In [26]:
print(f'Mean accuracy:      {resultados.mean():.4f}')
print(f'Standard deviation: {resultados.std():.4f}')

Mean accuracy:      0.8242
Standard deviation: 0.0568


## Overfitting and Dropout

**Apply 20% dropout to both hidden layers to reduce overfitting. How do the mean score and standard deviation change?**

Dropout disables 20% of the activations during training. Compare models using both the mean and standard deviation of the validation scores without assuming that dropout must improve the result.

In [27]:
def criar_rede_dropout():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
    return rede_neural

In [28]:
rede_neural = KerasClassifier(model=criar_rede_dropout, epochs=100, batch_size=10, verbose=0)

In [29]:
resultados = cross_val_score(estimator=rede_neural, X=X, y=y, cv=10, scoring='accuracy')

In [30]:
resultados

array([0.68421053, 0.84210526, 0.92982456, 0.78947368, 0.92982456,
       0.94736842, 0.9122807 , 0.94736842, 0.80701754, 0.96428571])

In [31]:
print(f'Mean accuracy with dropout:      {resultados.mean():.4f}')
print(f'Standard deviation with dropout: {resultados.std():.4f}')

Mean accuracy with dropout:      0.8754
Standard deviation with dropout: 0.0868


## Hyperparameter Tuning

Hyperparameter tuning compares configurations defined before training.
Examples include:

- Learning rate
- Number of trees in a random forest
- Number of layers or neurons in a neural network
- SVM kernel type

Common search strategies include:

- Grid Search
- Random Search
- Bayesian optimization

**Describe how the network is configured for hyperparameter tuning.**

The `criar_rede` function receives the parameters to compare. `KerasClassifier` adapts the network to the scikit-learn interface, and `GridSearchCV` evaluates each combination with five folds.

In [32]:
def criar_rede(optimizer, loss, kernel_initializer, activation, neurons):
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    rede_neural.compile(optimizer=optimizer, loss=loss, metrics=['binary_accuracy'])
    return rede_neural

In [33]:
rede_neural = KerasClassifier(model=criar_rede, verbose=0)

In [34]:
parametros = {
    'batch_size': [10, 30],
    'epochs': [50, 100],
    'model__optimizer': ['adam', 'sgd'],
    'model__loss': ['binary_crossentropy', 'hinge'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu', 'tanh'],
    'model__neurons': [16, 8]
}

In [35]:
# Use this configuration if the previous search is too expensive
parametros = {
    'batch_size': [10, 30],
    'epochs': [50],
    'model__optimizer': ['adam'],
    'model__loss': ['binary_crossentropy'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu'],
    'model__neurons': [16]
}

In [36]:
parametros

{'batch_size': [10, 30],
 'epochs': [50],
 'model__optimizer': ['adam'],
 'model__loss': ['binary_crossentropy'],
 'model__kernel_initializer': ['random_uniform', 'normal'],
 'model__activation': ['relu'],
 'model__neurons': [16]}

In [37]:
grid_search = GridSearchCV(estimator=rede_neural, param_grid=parametros,
                           scoring='accuracy', cv=5)

In [38]:
grid_search = grid_search.fit(X, y)

In [39]:
print(grid_search)

GridSearchCV(cv=5,
             estimator=KerasClassifier(model=<function criar_rede at 0x000001FAEEFB60C0>, verbose=0),
             param_grid={'batch_size': [10, 30], 'epochs': [50],
                         'model__activation': ['relu'],
                         'model__kernel_initializer': ['random_uniform',
                                                       'normal'],
                         'model__loss': ['binary_crossentropy'],
                         'model__neurons': [16], 'model__optimizer': ['adam']},
             scoring='accuracy')


In [40]:
melhores_parametros = grid_search.best_params_
print(melhores_parametros)

{'batch_size': 10, 'epochs': 50, 'model__activation': 'relu', 'model__kernel_initializer': 'random_uniform', 'model__loss': 'binary_crossentropy', 'model__neurons': 16, 'model__optimizer': 'adam'}


In [41]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao)

0.9033069399161621


**How could the network be improved further?**

Possible strategies include:
- **Expand the grid search space**: test more combinations of neurons, layers, optimizers, and learning rates.
- **Scale the features**: `StandardScaler` or `MinMaxScaler` can improve convergence.
- **Train for more epochs**: additional epochs may help when combined with regularization and validation-based stopping.
- **Add layers**: deeper architectures can represent more complex patterns.
- **Batch normalization**: stabilize activations and accelerate training.
- **Early stopping**: stop training when validation loss no longer improves.

# Saving the Trained Model

In [42]:
# Treina o modelo final com os melhores parâmetros antes de salvar
k.clear_session()
classificador = Sequential([
    tf.keras.layers.InputLayer(shape=(30,)),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])
otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
classificador.compile(optimizer=otimizador, loss='binary_crossentropy', metrics=['binary_accuracy'])
classificador.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100, verbose=0)

In [43]:
classificador.save('breast_cancer_classifier.keras')

In [44]:
classificador_novo = tf.keras.models.load_model('breast_cancer_classifier.keras')

In [45]:
classificador_novo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,357 (9.21 KB)

 Trainable params: 785 (3.07 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,572 (6.14 KB)